# Telco Customer Churn - Exploratory Data Analysis (EDA)
This notebook contains visualizations to analyze and understand the distribution of features in the Telco Customer Churn dataset.

**Note:** If you get an `ImportError` (e.g., `ModuleNotFoundError: No module named 'seaborn'`), please run the cell below to install the required libraries, then restart the kernel.

In [ ]:
# Run this cell to install the required libraries if they are missing
# %pip install pandas matplotlib seaborn

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for premium look
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Load the dataset
df = pd.read_csv('~/Downloads/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Clean 'TotalCharges' (it has empty spaces representing missing values for new customers with tenure = 0)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print(f"Dataset successfully loaded. Shape: {df.shape}")

## 1. Target Variable: Churn Distribution
Let's see the proportion of customers who churned vs. those who stayed.

In [ ]:
plt.figure(figsize=(6, 4.5))
ax = sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Distribution of Customer Churn', fontsize=14, pad=15)
plt.xlabel('Churn Status', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Add percentage labels on top of the bars
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    ax.annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height() + 50),
                ha='center', va='baseline', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Numerical Features Distribution
We will visualize the distribution of `tenure`, `MonthlyCharges`, and `TotalCharges` split by `Churn` to see how these numerical factors correlate with churn.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Tenure distribution
sns.histplot(data=df, x='tenure', hue='Churn', kde=True, multiple='stack', palette='crest', ax=axes[0])
axes[0].set_title('Distribution of Tenure (Months)', fontsize=13, pad=10)
axes[0].set_xlabel('Tenure (Months)')
axes[0].set_ylabel('Count')

# Monthly Charges distribution
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', kde=True, multiple='stack', palette='magma', ax=axes[1])
axes[1].set_title('Distribution of Monthly Charges', fontsize=13, pad=10)
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Count')

# Total Charges distribution
sns.histplot(data=df, x='TotalCharges', hue='Churn', kde=True, multiple='stack', palette='viridis', ax=axes[2])
axes[2].set_title('Distribution of Total Charges', fontsize=13, pad=10)
axes[2].set_xlabel('Total Charges ($)')
axes[2].set_ylabel('Count')

plt.suptitle('Distribution of Numerical Features by Churn Status', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 3. Key Categorical Features Distribution
Let's look at key services and contract options to see which categories show higher churn rates.

In [ ]:
categorical_cols = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    sns.countplot(data=df, x=col, hue='Churn', palette='Set2', ax=axes[i])
    axes[i].set_title(f'Churn Distribution by {col}', fontsize=13, pad=10)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')
    # Rotate the tick labels safely without overwriting their text values
    plt.setp(axes[i].get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Distribution of Key Categorical Features by Churn Status', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing
Before training a machine learning model, we need to prepare the data by handling missing values and encoding categorical features into numerical values.

In [ ]:
# Reload the original data to perform clean preprocessing steps
df_clean = pd.read_csv('~/Downloads/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print("--- 1. Handling Missing Values in TotalCharges ---")
# TotalCharges column has empty string values for new customers with tenure = 0.
# We will convert them to numeric (turning empty strings to NaN) and fill NaN values with 0.
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'].str.strip(), errors='coerce')
missing_count = df_clean['TotalCharges'].isnull().sum()
print(f"Number of missing values in TotalCharges before handling: {missing_count}")

# Fill missing values with 0
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0)
print(f"Number of missing values in TotalCharges after handling: {df_clean['TotalCharges'].isnull().sum()}")

print("\n--- 2. Categorical Encoding ---")
# Let's apply Label Encoding for binary features (gender, Partner, Churn)
print("Applying Label Encoding for binary features (gender, Partner, Churn)...")
df_clean['gender'] = df_clean['gender'].map({'Female': 0, 'Male': 1})
df_clean['Partner'] = df_clean['Partner'].map({'No': 0, 'Yes': 1})
df_clean['Churn'] = df_clean['Churn'].map({'No': 0, 'Yes': 1})

# Let's apply One-Hot Encoding for InternetService
print("Applying One-Hot Encoding for InternetService...")
# Using pd.get_dummies (setting dtype=int for clean 1/0 representation)
df_clean = pd.get_dummies(df_clean, columns=['InternetService'], prefix='Internet', dtype=int)

# Preview the processed dataframe columns
print("\nEncoded features overview:")
print(df_clean[['gender', 'Partner', 'Internet_DSL', 'Internet_Fiber optic', 'Internet_No', 'TotalCharges', 'Churn']].head())